In [757]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import preprocessor as pp

In [759]:
import importlib
importlib.reload(pp)

<module 'preprocessor' from '/Users/victorli/Desktop/vsc/Programming with Data/DS2500/Battery-RUL-Analysis/preprocessor.py'>

## Preprocessing and Splitting

In [762]:
path = 'cleaned_dataset/'

In [764]:
df = pp.read_clean_file(path)
df

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,2010-07-21 15:00:35.093000,4,B0047,0,1,00001.csv,1.674305,NaN,NaN
1,impedance,2010-07-21 16:53:45.968000,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,2010-07-21 17:25:40.670999,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,2010-07-21 20:31:05.000000,24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,2010-07-21 21:02:56.984000,4,B0047,4,5,00005.csv,1.524366,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
7560,impedance,2010-09-30 07:36:45.045999,24,B0055,247,7561,07561.csv,NaN,0.0968087979207628,0.15489738203707232
7561,discharge,2010-09-30 08:08:36.328000,4,B0055,248,7562,07562.csv,1.020138,NaN,NaN
7562,charge,2010-09-30 08:48:54.250000,4,B0055,249,7563,07563.csv,NaN,NaN,NaN
7563,discharge,2010-09-30 11:50:17.687000,4,B0055,250,7564,07564.csv,0.990759,NaN,NaN


In [766]:
BATTERIES = sorted(df['battery_id'].value_counts().index.tolist())
print(len(BATTERIES))
bat_tr, bat_te = train_test_split(BATTERIES, test_size=0.28, random_state=42)
print(bat_tr)
print(bat_te)
df_tr = df[df['battery_id'].isin(bat_tr)]
df_te = df[df['battery_id'].isin(bat_te)]
dis_tr = pp.get_discharges_phyiscs(path, df_tr)
dis_te = pp.get_discharges_phyiscs(path, df_te)

34
['B0005', 'B0025', 'B0039', 'B0040', 'B0026', 'B0034', 'B0032', 'B0006', 'B0007', 'B0053', 'B0018', 'B0052', 'B0046', 'B0054', 'B0045', 'B0041', 'B0048', 'B0027', 'B0043', 'B0056', 'B0028', 'B0031', 'B0036', 'B0051']
['B0038', 'B0042', 'B0050', 'B0049', 'B0029', 'B0047', 'B0044', 'B0033', 'B0055', 'B0030']


In [767]:
bat_tr, bat_val = train_test_split(BATTERIES,test_size=0.2,random_state=21)
print(bat_tr)
print(bat_val)

df_tr = df[df['battery_id'].isin(bat_tr)]
df_val = df[df['battery_id'].isin(bat_val)]
dis_tr = pp.get_discharges_phyiscs(path, df_tr)
dis_val = pp.get_discharges_phyiscs(path, df_val)
dis_tr = pp.merge_impedance_with_discharges(df_tr, dis_tr, path)
dis_val = pp.merge_impedance_with_discharges(df_val, dis_val, path)



['B0055', 'B0026', 'B0042', 'B0045', 'B0028', 'B0050', 'B0032', 'B0041', 'B0027', 'B0053', 'B0044', 'B0051', 'B0056', 'B0033', 'B0036', 'B0029', 'B0031', 'B0040', 'B0007', 'B0018', 'B0054', 'B0052', 'B0039', 'B0025', 'B0047', 'B0038', 'B0030']
['B0006', 'B0043', 'B0049', 'B0005', 'B0034', 'B0048', 'B0046']


In [769]:
print(df.dtypes)

type                           object
start_time             datetime64[ns]
ambient_temperature             int64
battery_id                     object
test_id                         int64
uid                             int64
filename                       object
Capacity                      float64
Re                             object
Rct                            object
dtype: object


In [739]:
pp.impedance_features(path, '00002.csv')

{'Battery_impedance_real_mean': 0.20667752523246005,
 'Battery_impedance_mag_mean': 0.21392855186944962,
 'Rectified_impedance_real_mean': 0.08360003221854975,
 'Rectified_impedance_phase_mean': -0.07934092705559784}

In [741]:
df_dis_tr = pp.merge_impedance_with_discharges(df_tr, dis_tr, path)
df_dis_tr

,start_time,ambient_temperature,battery_id,uid,filename,Capacity,cycle_number,SOH,EOL_cycle,RUL,...,capacity_Ah,capacity_ratio,voltage_drop,cycle_sqrt,cycle_sq,cycle_log,Battery_impedance_real_mean,Battery_impedance_mag_mean,Rectified_impedance_real_mean,Rectified_impedance_phase_mean
0,2008-04-02 15:25:41.593000,24,B0007,5738,05738.csv,1.891052,1,0.945526,168,167,...,1.919029,1.014795,1.137247,1.000000,1,0.693147,0.165188,0.185893,0.051705,-0.017040
1,2008-04-02 19:43:48.405999,24,B0007,5740,05740.csv,1.880637,2,0.940319,168,166,...,1.908615,1.014877,1.119031,1.414214,4,1.098612,0.165188,0.185893,0.051705,-0.017040
2,2008-04-03 00:01:06.687000,24,B0007,5742,05742.csv,1.880663,3,0.940331,168,165,...,1.908615,1.014863,1.135015,1.732051,9,1.386294,0.165188,0.185893,0.051705,-0.017040
3,2008-04-03 04:16:37.375000,24,B0007,5744,05744.csv,1.880771,4,0.940385,168,164,...,1.908792,1.014899,1.158521,2.000000,16,1.609438,0.165188,0.185893,0.051705,-0.017040
4,2008-04-03 08:33:25.702999,24,B0007,5746,05746.csv,1.879451,5,0.939725,168,163,...,1.907301,1.014818,1.155297,2.236068,25,1.791759,0.165188,0.185893,0.051705,-0.017040
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1939,2010-09-30 08:08:36.328000,4,B0056,7310,07310.csv,1.137273,101,0.568637,102,0,...,1.140995,1.003273,0.449468,10.049876,10201,4.624973,0.262476,0.266944,0.131623,-0.035036
1940,2010-09-30 11:50:17.687000,4,B0054,7059,07059.csv,0.837392,102,0.418696,103,0,...,1.223278,1.460819,0.565554,10.099505,10404,4.634729,0.257711,0.262117,0.129947,-0.037604
1941,2010-09-30 11:50:17.687000,4,B0053,6806,06806.csv,1.010274,55,0.505137,56,0,...,1.308950,1.295638,2.258601,7.416198,3025,4.025352,0.288265,0.294110,0.130685,-0.030323
1942,2010-09-30 11:50:17.687000,4,B0055,7564,07564.csv,0.990759,102,0.495380,102,0,...,1.108000,1.118335,0.468319,10.099505,10404,4.634729,0.268506,0.272732,0.124002,-0.029440


## Predicting Capacity

In [616]:
features = ['ambient_temperature', 
            'cycle_number',
            'mean_voltage',
            'max_voltage',
            'min_voltage',
            'mean_current',
            'max_current',
            'mean_temperature',
            'max_temperature',
            'discharge_time']
target = 'Capacity'
disx_tr = dis_tr[features].values
disy_tr = dis_tr[target].values
disx_val = dis_val[features].values
disy_val = dis_val[target].values

In [782]:
features_physics = ['ambient_temperature', 
            'cycle_number',
            'mean_voltage',
            'max_voltage',
            'min_voltage',
            'mean_current',
            'max_current',
            'mean_temperature',
            'max_temperature',
            'discharge_time',
            'r_internal', 
            'mean_dvdt', 
            'voltage_drop']
target = 'Capacity'

disx_tr = dis_tr[features_physics].values
disy_tr = dis_tr[target].values
disx_val = dis_val[features_physics].values
disy_val = dis_val[target].values

In [784]:
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=12,
    random_state=21
)
rf_model.fit(disx_tr, disy_tr)

pred_rf = rf_model.predict(disx_val)

print("RF MAE:", mean_absolute_error(disy_val, pred_rf))
print("RF MSE:", mean_squared_error(disy_val, pred_rf))
print("RF R2:", r2_score(disy_val, pred_rf))

'''
trained on 'features':
RF MAE: 0.057776649944968075
RF MSE: 0.006399883004647191
RF R2: 0.9574231322107574

before impedence features:
trained on 'features_physics':
RF MAE: 0.052649303739642155
RF MSE: 0.005196369994694295
RF R2: 0.9654298120625905

after:
'''

RF MAE: 0.052539270885779116
RF MSE: 0.005022797323523483
RF R2: 0.9665845489018263


"\ntrained on 'features':\nRF MAE: 0.057776649944968075\nRF MSE: 0.006399883004647191\nRF R2: 0.9574231322107574\n\nbefore impedence features:\ntrained on 'features_physics':\nRF MAE: 0.052649303739642155\nRF MSE: 0.005196369994694295\nRF R2: 0.9654298120625905\n\nafter:\n"

In [785]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=21
)

xgb_model.fit(disx_tr, disy_tr)

pred_xgb = xgb_model.predict(disx_val)

print("XGB MAE:", mean_absolute_error(disy_val, pred_xgb))
print("XGB RMSE:", np.sqrt(mean_squared_error(disy_val, pred_xgb)))
print("XGB R2:", r2_score(disy_val, pred_xgb))

'''
trained on 'features':
XGB MAE: 0.06543340493941216
XGB RMSE: 0.09152594937403981
XGB R2: 0.9442698568019331

trained on 'features_physics':
XGB MAE: 0.049791549235134916
XGB RMSE: 0.06420838974522229
XGB R2: 0.9725725626750873
'''

XGB MAE: 0.04802586560446461
XGB RMSE: 0.06250523406268285
XGB R2: 0.9740083143556363


"\ntrained on 'features':\nXGB MAE: 0.06543340493941216\nXGB RMSE: 0.09152594937403981\nXGB R2: 0.9442698568019331\n\ntrained on 'features_physics':\nXGB MAE: 0.049791549235134916\nXGB RMSE: 0.06420838974522229\nXGB R2: 0.9725725626750873\n"

## Predicting RUL

In [644]:
target = 'RUL'
rulx_tr = dis_tr[features_physics].values
ruly_tr = dis_tr[target].values
rulx_val = dis_val[features_physics].values
ruly_val = dis_val[target].values

In [646]:
ruly_tr

array([167, 166, 165, ...,   0,   0,   0])

In [648]:

xgb_model = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.7,
    colsample_bytree=0.8,
)

xgb_model.fit(rulx_tr, ruly_tr)
rul_pred_rf = xgb_model.predict(rulx_val)
print("XGB MAE:", mean_absolute_error(ruly_val, rul_pred_rf))
print("XGB MSE:", mean_squared_error(ruly_val, rul_pred_rf))
print("XGB R2:", r2_score(ruly_val, rul_pred_rf))

XGB MAE: 9.163642476150127
XGB MSE: 270.6516297210585
XGB R2: 0.7361499378734815


'\ncycle_sqrt:\nXGB MAE: 9.163642476150127\nXGB MSE: 270.6516297210585\nXGB R2: 0.7361499378734815\ncycle_sq:\nXGB MAE: 9.163642476150127\nXGB MSE: 270.6516297210585\nXGB R2: 0.7361499378734815\ncycle_log:\nXGB MAE: 9.163642476150127\nXGB MSE: 270.6516297210585\nXGB R2: 0.7361499378734815\n'

In [649]:
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=12,
    random_state=21
)
rf_model.fit(rulx_tr, ruly_tr)

rul_pred_rf = rf_model.predict(rulx_val)

print("RF MAE:", mean_absolute_error(ruly_val, rul_pred_rf))
print("RF MSE:", mean_squared_error(ruly_val, rul_pred_rf))
print("RF R2:", r2_score(ruly_val, rul_pred_rf))

RF MAE: 9.20780640660513
RF MSE: 318.26846539654207
RF R2: 0.689729729489024


'\ncycle_sqrt:\nRF MAE: 9.2093270514762\nRF MSE: 318.81689140043557\nRF R2: 0.6891950856173126\ncycle_sq:\nRF MAE: 9.2093270514762\nRF MSE: 318.81689140043557\nRF R2: 0.6891950856173126\ncycle_log:\nRF MAE: 9.209730483102781\nRF MSE: 319.48734320334205\nRF R2: 0.6885414824964592\n'

In [360]:
#window size / number of past cycles to see
window = 10

In [362]:
def build_sequences(df_disch, window=window):
    df = df_disch.sort_values(['battery_id','cycle_number'])
    X_seq = []
    y_seq = []

    for bid, g in df.groupby('battery_id'):
        g = g.dropna(subset=[target])
        g_feat = g[features].values
        g_cap  = g[target].values

        for i in range(window, len(g)):
            X_seq.append(g_feat[i-window:i])  
            y_seq.append(g_cap[i])            

    return np.array(X_seq), np.array(y_seq)

Xtr_seq, ytr_seq = build_sequences(dis_tr)
Xval_seq, yval_seq = build_sequences(dis_val)

In [364]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

n_feats = len(features)

model = Sequential([
    LSTM(64, return_sequences=False, input_shape=(window, n_feats)),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

history = model.fit(
    Xtr_seq, ytr_seq,
    validation_data=(Xval_seq, yval_seq),
    epochs=40,
    batch_size=32,
    verbose=1
)

pred_lstm = model.predict(Xval_seq).flatten()

print("LSTM MAE:", mean_absolute_error(yval_seq, pred_lstm))
print("LSTM RMSE:", np.sqrt(mean_squared_error(yval_seq, pred_lstm)))

Epoch 1/40


/opt/anaconda3/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.9302 - val_loss: 0.1168
Epoch 2/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1792 - val_loss: 0.1134
Epoch 3/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1721 - val_loss: 0.1028
Epoch 4/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1396 - val_loss: 0.0728
Epoch 5/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1022 - val_loss: 0.0482
Epoch 6/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0864 - val_loss: 0.0452
Epoch 7/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0640 - val_loss: 0.0495
Epoch 8/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0469 - val_loss: 0.0443
Epoch 9/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0503 - val_loss: 0.0531
Epoch 10/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0450 - val_loss: 0.0562
Epoch 11/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0447 - val_loss: 0.0588
Epoch 12/40
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0532 - val_loss: 0.0376


In [ ]:
def extract_